In [8]:
# Cell 1: Imports

# Note: the first run of this cell may take several minutes to complete

from dotenv import load_dotenv
import os
import sys
import importlib
import glob
from IPython.display import HTML, display
import pandas as pd
import google.generativeai as genai
import ipywidgets

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

load_dotenv()

from onetwo import ot
from onetwo.backends import gemini_api

# Import local modules
try:
    import data_utils
    import colab_utils
    import prompt_templates
    import phia_agent

    importlib.reload(data_utils)
    importlib.reload(colab_utils)
    importlib.reload(prompt_templates)
    importlib.reload(phia_agent)
    print("Modules imported/reloaded.")
except ImportError as e:
    print(f"Error importing local modules: {e}")
    print("Please ensure data_utils.py, colab_utils.py, prompt_templates.py, and phia_agent.py are in the same directory as the notebook.")

from data_utils import load_persona
from colab_utils import format_react_state_html
from phia_agent import get_react_agent, QUESTION_PREFIX

Modules imported/reloaded.


In [9]:
# Cell 2: Set API Keys

# Replace with your API keys below.
#
# WARNING: Don't leave your API keys here when committing 
# this notebook into any public code repos!
google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Set the Google API key below and when setting up backend, 
# the tavily API key will be set when creating the agent.
genai.configure(api_key=google_api_key)

In [10]:
# Cell 3: Setup LLM Backend

llm_engine = gemini_api.GeminiAPI(
    generate_model_name="models/gemini-2.5-flash-lite", # Can replace with a better model, if desired.
                                                 # Note: due to the nature of our few_shots
                                                 # examples that were designed with Gemini 1.5
                                                 # models in mind, agent behavior with Gemini 
                                                 # 2.0 or 2.5 models may have errors related
                                                 # to output formatting.
    api_key=google_api_key,
    temperature=0.0,
)
llm_engine.register()
print("LLM Backend Registered.")

LLM Backend Registered.


E0000 00:00:1760474466.369522  844569 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [11]:
# Cell 4: Load Data

summary_path = os.path.join("synthetic_wearable_users", "summary_df_502.csv")
activities_path = os.path.join("synthetic_wearable_users", "exercise_df_502.csv")

print(f"Loading data from: {summary_path} and {activities_path}")

try:
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"Cannot find summary file: {summary_path}")
    if not os.path.exists(activities_path):
        raise FileNotFoundError(f"Cannot find activities file: {activities_path}")

    summary_df, activities_df, profile_df = load_persona(
        summary_path=summary_path,
        activities_path=activities_path,
        enforce_schema=True,
        temporally_localize="today"
    )
    print("Data Loaded Successfully:")
    display(f"Summary DF: {summary_df.shape}")
    display(summary_df.head(1))
    display(f"Activities DF: {activities_df.shape}")
    display(activities_df.head(1))
    display(f"Profile DF: {profile_df.shape}")
    display(profile_df.head(1))
except Exception as e:
    print(f"Error loading data: {e}")

Loading data from: synthetic_wearable_users/summary_df_502.csv and synthetic_wearable_users/exercise_df_502.csv
Data Loaded Successfully:


'Summary DF: (29, 20)'

,datetime,resting_heart_rate,heart_rate_variability,fatburn_active_zone_minutes,cardio_active_zone_minutes,peak_active_zone_minutes,active_zone_minutes,steps,rem_sleep_minutes,deep_sleep_minutes,awake_minutes,light_sleep_minutes,sleep_minutes,bed_time,wake_up_time,stress_management_score,deep_sleep_percent,rem_sleep_percent,awake_percent,light_sleep_percent
datetime,,,,,,,,,,,,,,,,,,,,
2025-09-14,2025-09-14,67.77,7.11,17.59,14.81,1.11,33.51,12637.39,61.79,85.32,49.59,194.07,390.77,2025-09-14 21:56:34,2025-09-15 04:27:20,81.46,21.83,15.81,12.69,49.66


'Activities DF: (4, 11)'

,startTime,endTime,activityName,distance,duration,elevationGain,averageHeartRate,calories,steps,activeZoneMinutes,speed
startTime,,,,,,,,,,,
2025-09-26 09:06:00,2025-09-26 09:06:00,2025-09-26 09:32:00,Outdoor Bike,0.0,25.62,30.48,90.0,136.0,0.0,0.0,0.0


'Profile DF: (1, 6)'

,age,gender,averageDailySteps,elderly,height_cm,weight_kg
0,52,Male,10121,No,184,81


In [12]:
# Cell 5: Load Exemplars

exemplar_dir = "few_shots"
exemplar_pattern = os.path.join(exemplar_dir, "*.ipynb")
final_exemplar_paths = glob.glob(exemplar_pattern)

if not final_exemplar_paths:
    print(f"Warning: No exemplar notebooks found in {exemplar_dir}/")
else:
    print(f"Found {len(final_exemplar_paths)} exemplar notebooks:")
    for path in final_exemplar_paths:
        print(f"  - {path}")

Found 46 exemplar notebooks:
  - few_shots/sleep_time_after_last_bike_ride.ipynb
  - few_shots/exercise_effects_on_sleep.ipynb
  - few_shots/manage_stress.ipynb
  - few_shots/days_slept_better.ipynb
  - few_shots/greeting.ipynb
  - few_shots/consistent_weekly_fitness.ipynb
  - few_shots/sleep_fastest_run.ipynb
  - few_shots/overall_health.ipynb
  - few_shots/optimize_exercise_and_wellness.ipynb
  - few_shots/correct_form.ipynb
  - few_shots/running_compared_olympian.ipynb
  - few_shots/awake_percent_standard_deviation_7_days.ipynb
  - few_shots/heart_rate_health_effects.ipynb
  - few_shots/workout_frequency.ipynb
  - few_shots/worst_HRV_last_month.ipynb
  - few_shots/10k_adjust_run_goal.ipynb
  - few_shots/sleep_stages_resting_heart_rate.ipynb
  - few_shots/HRV_bedtime.ipynb
  - few_shots/average_HR_and_calorie_burn_activities.ipynb
  - few_shots/cardiovascular_health.ipynb
  - few_shots/REM_sleep_percentage_yesterday.ipynb
  - few_shots/avg_rhr_in_two_weeks.ipynb
  - few_shots/highest

In [13]:
# Cell 6: Create Agent

try:
    agent = get_react_agent(
        summary_df=summary_df,
        activities_df=activities_df,
        profile_df=profile_df,
        example_files=final_exemplar_paths,
        tavily_api_key=tavily_api_key,
        use_mock_search=False
    )
    print("Agent Created Successfully!")
except Exception as e:
    print(f"Error creating agent: {e}")

Processing file: few_shots/sleep_time_after_last_bike_ride.ipynb
Processing file: few_shots/exercise_effects_on_sleep.ipynb
Processing file: few_shots/manage_stress.ipynb
Processing file: few_shots/days_slept_better.ipynb
Processing file: few_shots/greeting.ipynb
Processing file: few_shots/consistent_weekly_fitness.ipynb
Processing file: few_shots/sleep_fastest_run.ipynb
Processing file: few_shots/overall_health.ipynb
Processing file: few_shots/optimize_exercise_and_wellness.ipynb
Processing file: few_shots/correct_form.ipynb
Processing file: few_shots/running_compared_olympian.ipynb
Processing file: few_shots/awake_percent_standard_deviation_7_days.ipynb
Processing file: few_shots/heart_rate_health_effects.ipynb
Processing file: few_shots/workout_frequency.ipynb
Processing file: few_shots/worst_HRV_last_month.ipynb
Processing file: few_shots/10k_adjust_run_goal.ipynb
Processing file: few_shots/sleep_stages_resting_heart_rate.ipynb
Processing file: few_shots/HRV_bedtime.ipynb
Processin

In [17]:
# Cell 7: Ask a Question

# Define your question below
question = "How many steps did I take in the last 21 days?"
full_question = QUESTION_PREFIX + question

try:
    final_answer, final_state = ot.run(
        agent(inputs=full_question, return_final_state=True)
    )

    print("\n--- Agent Trace ---")
    html_output = format_react_state_html(final_state)
    display(HTML(html_output))

    print("\n--- Final Answer ---")
    if final_answer:
        display(HTML(str(final_answer).replace('\n', '<br>')))
    else:
        display("No final answer was returned by the agent.")

except Exception as e:
    print(f"An error occurred during agent execution: {e}")


--- Agent Trace ---



--- Final Answer ---
